# <h1><b> NLP: TEXT TAGGING PRACTICAL <b><h1>

In [355]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import spacy
import re
import pandas as pd
import matplotlib.pyplot as plt

### Load data

In [356]:
pd.set_option("display.max_colwidth", None)
bbc_df = pd.read_csv(r"txt_files\bbc_news.csv")

In [357]:
bbc_df.head(3)

,Unnamed: 0,index,title,pubDate,guid,link,description
0,0,6684,Can I refuse to work?,"Wed, 10 Aug 2022 15:46:18 GMT",https://www.bbc.co.uk/news/business-62147992,https://www.bbc.co.uk/news/business-62147992?at_medium=RSS&at_campaign=KARANGA,"With much of the UK enduring another period of hot weather, some workers will face very high temperatures."
1,1,9267,'Liz Truss the Brief?' World reacts to UK political turmoil,"Mon, 17 Oct 2022 11:35:12 GMT",https://www.bbc.co.uk/news/world-63285480,https://www.bbc.co.uk/news/world-63285480?at_medium=RSS&at_campaign=KARANGA,The UK's political chaos has been watched around the world - and commentators haven't held back.
2,2,7387,Rationing energy is nothing new for off-grid community,"Wed, 31 Aug 2022 05:20:18 GMT",https://www.bbc.co.uk/news/uk-scotland-highlands-islands-62686916,https://www.bbc.co.uk/news/uk-scotland-highlands-islands-62686916?at_medium=RSS&at_campaign=KARANGA,Scoraig in the north west Highlands has long had to make careful use of its electricity supply.


In [358]:
bbc_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   Unnamed: 0   1000 non-null   int64
 1   index        1000 non-null   int64
 2   title        1000 non-null   str  
 3   pubDate      1000 non-null   str  
 4   guid         1000 non-null   str  
 5   link         1000 non-null   str  
 6   description  1000 non-null   str  
dtypes: int64(2), str(5)
memory usage: 54.8 KB


In [359]:
titles_df = pd.DataFrame(bbc_df["title"])
titles_df.head(10)

,title
0,Can I refuse to work?
1,'Liz Truss the Brief?' World reacts to UK political turmoil
2,Rationing energy is nothing new for off-grid community
3,The hunt for superyachts of sanctioned Russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 seconds
5,Red Bull found guilty of breaking Formula 1's budget cap
6,World Triathlon Championship Series: Flora Duffy beats Georgia Taylor-Brown to women's title
7,Terry Hall: Coventry scooter ride-out pays tribute to singer
8,Post Office and Fujitsu to face inquiry over Horizon scandal
9,'Pavement parking frightens me'


### Clean data

In [360]:
# lowercase
titles_df["lowercase"] = titles_df["title"].str.lower()
titles_df.head(3)

,title,lowercase
0,Can I refuse to work?,can i refuse to work?
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community


In [361]:
# stop word removal
en_stopwords = stopwords.words("english")
en_stopwords

['a',
 'about',
 'above',
 'after',
 'again',
 'against',
 'ain',
 'all',
 'am',
 'an',
 'and',
 'any',
 'are',
 'aren',
 "aren't",
 'as',
 'at',
 'be',
 'because',
 'been',
 'before',
 'being',
 'below',
 'between',
 'both',
 'but',
 'by',
 'can',
 'couldn',
 "couldn't",
 'd',
 'did',
 'didn',
 "didn't",
 'do',
 'does',
 'doesn',
 "doesn't",
 'doing',
 'don',
 "don't",
 'down',
 'during',
 'each',
 'few',
 'for',
 'from',
 'further',
 'had',
 'hadn',
 "hadn't",
 'has',
 'hasn',
 "hasn't",
 'have',
 'haven',
 "haven't",
 'having',
 'he',
 "he'd",
 "he'll",
 'her',
 'here',
 'hers',
 'herself',
 "he's",
 'him',
 'himself',
 'his',
 'how',
 'i',
 "i'd",
 'if',
 "i'll",
 "i'm",
 'in',
 'into',
 'is',
 'isn',
 "isn't",
 'it',
 "it'd",
 "it'll",
 "it's",
 'its',
 'itself',
 "i've",
 'just',
 'll',
 'm',
 'ma',
 'me',
 'mightn',
 "mightn't",
 'more',
 'most',
 'mustn',
 "mustn't",
 'my',
 'myself',
 'needn',
 "needn't",
 'no',
 'nor',
 'not',
 'now',
 'o',
 'of',
 'off',
 'on',
 'once',
 'on

In [362]:
titles_df["rm_stopwords"] = titles_df["lowercase"].apply(lambda data: " ".join([word for word in data.split() if word not in en_stopwords]))
titles_df.head(5)

,title,lowercase,rm_stopwords
0,Can I refuse to work?,can i refuse to work?,refuse work?
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community
3,The hunt for superyachts of sanctioned Russian oligarchs,the hunt for superyachts of sanctioned russian oligarchs,hunt superyachts sanctioned russian oligarchs
4,Platinum Jubilee: 70 years of the Queen in 70 seconds,platinum jubilee: 70 years of the queen in 70 seconds,platinum jubilee: 70 years queen 70 seconds


In [363]:
# punctation removal
titles_df["rm_punctuation"] = titles_df["rm_stopwords"].apply(lambda x: re.sub(r"[^\w\s]", "", x))
titles_df.head(3)


,title,lowercase,rm_stopwords,rm_punctuation
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil,liz truss brief world reacts uk political turmoil
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community


In [364]:
# tokenize
titles_df["token_raw"] = titles_df.apply(lambda x: word_tokenize(x["title"]), axis=1)
titles_df["token_clean"] = titles_df.apply(lambda x: word_tokenize(x["rm_punctuation"]), axis=1)
titles_df.head(3)


,title,lowercase,rm_stopwords,rm_punctuation,token_raw,token_clean
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts, to, UK, political, turmoil]","[liz, truss, brief, world, reacts, uk, political, turmoil]"
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off-grid, community]","[rationing, energy, nothing, new, offgrid, community]"


In [365]:
# lemmatizing 
lem = WordNetLemmatizer()
titles_df["lemmatizing"] = titles_df.apply(lambda x: [lem.lemmatize(word) for word in x["token_clean"]], axis=1)
titles_df.head(3)


,title,lowercase,rm_stopwords,rm_punctuation,token_raw,token_clean,lemmatizing
0,Can I refuse to work?,can i refuse to work?,refuse work?,refuse work,"[Can, I, refuse, to, work, ?]","[refuse, work]","[refuse, work]"
1,'Liz Truss the Brief?' World reacts to UK political turmoil,'liz truss the brief?' world reacts to uk political turmoil,'liz truss brief?' world reacts uk political turmoil,liz truss brief world reacts uk political turmoil,"['Liz, Truss, the, Brief, ?, ', World, reacts, to, UK, political, turmoil]","[liz, truss, brief, world, reacts, uk, political, turmoil]","[liz, truss, brief, world, reacts, uk, political, turmoil]"
2,Rationing energy is nothing new for off-grid community,rationing energy is nothing new for off-grid community,rationing energy nothing new off-grid community,rationing energy nothing new offgrid community,"[Rationing, energy, is, nothing, new, for, off-grid, community]","[rationing, energy, nothing, new, offgrid, community]","[rationing, energy, nothing, new, offgrid, community]"


In [366]:
# create lists for just our tokens
tokens_raw_lst = sum(titles_df["token_raw"], [])
tokens_clean_lst = sum(titles_df["lemmatizing"], [])

In [367]:
tokens_raw_lst

['Can',
 'I',
 'refuse',
 'to',
 'work',
 '?',
 "'Liz",
 'Truss',
 'the',
 'Brief',
 '?',
 "'",
 'World',
 'reacts',
 'to',
 'UK',
 'political',
 'turmoil',
 'Rationing',
 'energy',
 'is',
 'nothing',
 'new',
 'for',
 'off-grid',
 'community',
 'The',
 'hunt',
 'for',
 'superyachts',
 'of',
 'sanctioned',
 'Russian',
 'oligarchs',
 'Platinum',
 'Jubilee',
 ':',
 '70',
 'years',
 'of',
 'the',
 'Queen',
 'in',
 '70',
 'seconds',
 'Red',
 'Bull',
 'found',
 'guilty',
 'of',
 'breaking',
 'Formula',
 '1',
 "'s",
 'budget',
 'cap',
 'World',
 'Triathlon',
 'Championship',
 'Series',
 ':',
 'Flora',
 'Duffy',
 'beats',
 'Georgia',
 'Taylor-Brown',
 'to',
 'women',
 "'s",
 'title',
 'Terry',
 'Hall',
 ':',
 'Coventry',
 'scooter',
 'ride-out',
 'pays',
 'tribute',
 'to',
 'singer',
 'Post',
 'Office',
 'and',
 'Fujitsu',
 'to',
 'face',
 'inquiry',
 'over',
 'Horizon',
 'scandal',
 "'Pavement",
 'parking',
 'frightens',
 'me',
 "'",
 'UK',
 'interest',
 'rates',
 ':',
 'How',
 'will',
 'the'

In [368]:
tokens_clean_lst

['refuse',
 'work',
 'liz',
 'truss',
 'brief',
 'world',
 'reacts',
 'uk',
 'political',
 'turmoil',
 'rationing',
 'energy',
 'nothing',
 'new',
 'offgrid',
 'community',
 'hunt',
 'superyachts',
 'sanctioned',
 'russian',
 'oligarch',
 'platinum',
 'jubilee',
 '70',
 'year',
 'queen',
 '70',
 'second',
 'red',
 'bull',
 'found',
 'guilty',
 'breaking',
 'formula',
 '1',
 'budget',
 'cap',
 'world',
 'triathlon',
 'championship',
 'series',
 'flora',
 'duffy',
 'beat',
 'georgia',
 'taylorbrown',
 'womens',
 'title',
 'terry',
 'hall',
 'coventry',
 'scooter',
 'rideout',
 'pay',
 'tribute',
 'singer',
 'post',
 'office',
 'fujitsu',
 'face',
 'inquiry',
 'horizon',
 'scandal',
 'pavement',
 'parking',
 'frightens',
 'me',
 'uk',
 'interest',
 'rate',
 'rise',
 'affect',
 'high',
 'could',
 'go',
 'stayed',
 'storm',
 'happens',
 'now',
 'six',
 'nation',
 'scotland',
 'best',
 'since',
 '99',
 'beat',
 'best',
 'ireland',
 'ever',
 'long',
 'liz',
 'truss',
 'survive',
 'prime',
 'm

### POS Tagging

In [369]:
nlp = spacy.load('en_core_web_sm')

In [370]:
# create a spacy doc from our raw text - better for pos tagging
spacy_doc = nlp(' '.join(tokens_raw_lst))

In [371]:
# extract the tokens and pos tags into a dataframe
pos_df = pd.DataFrame(columns=["tokens", "pos_tags"])
for token in spacy_doc:
    pos_df = pd.concat([pos_df, pd.DataFrame.from_records([{'tokens': token.text, "pos_tags": token.pos_}])], ignore_index=True)
pos_df.head(20)

,tokens,pos_tags
0,Can,AUX
1,I,PRON
2,refuse,VERB
3,to,PART
4,work,VERB
5,?,PUNCT
6,',PUNCT
7,Liz,PROPN
8,Truss,PROPN
9,the,DET


In [372]:
# token frequency count
pos_df_count = pos_df.groupby(["tokens", "pos_tags"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
pos_df_count.head(15)

,tokens,pos_tags,counts
95,:,PUNCT,543
8,',PUNCT,300
2897,in,ADP,187
4082,to,PART,175
3268,of,ADP,172
22,-,PUNCT,166
4043,the,DET,163
1856,and,CCONJ,147
15,'s,PART,143
97,?,PUNCT,130


In [373]:
pos_frequency = pos_df_count.groupby(["pos_tags"])["tokens"].count().sort_values(ascending=False)
pos_frequency.head(15)

pos_tags
NOUN     1487
PROPN    1346
VERB      759
ADJ       390
NUM        87
ADV        76
AUX        49
ADP        48
PRON       45
DET        21
SCONJ      21
PUNCT      17
PART        7
CCONJ       5
INTJ        4
Name: tokens, dtype: int64

In [374]:
# most common nouns
nouns = pos_df_count[pos_df_count.pos_tags == "NOUN"]
nouns.head(15)

,tokens,pos_tags,counts
4267,war,NOUN,35
3552,record,NOUN,15
4356,year,NOUN,14
4316,win,NOUN,14
3416,police,NOUN,14
3061,living,NOUN,13
4009,tax,NOUN,13
3368,people,NOUN,12
2326,day,NOUN,12
4357,years,NOUN,11


In [375]:
# most common verbs
verbs = pos_df_count[pos_df_count.pos_tags == "VERB"]
verbs.head(15)

,tokens,pos_tags,counts
3687,says,VERB,30
9,',VERB,14
2670,found,VERB,13
4317,win,VERB,12
4324,wins,VERB,10
2713,get,VERB,9
2388,dies,VERB,9
3108,make,VERB,8
2982,killed,VERB,8
3686,say,VERB,8


In [376]:
# most common adjectives
adjectives = pos_df_count[pos_df_count.pos_tags == "ADV"]
adjectives.head(15)

,tokens,pos_tags,counts
1938,back,ADV,8
3825,so,ADV,7
3260,now,ADV,5
3901,still,ADV,5
1823,again,ADV,4
2668,forward,ADV,4
3001,last,ADV,4
3069,long,ADV,4
4089,too,ADV,3
3545,really,ADV,3


In [377]:
from IPython.display import HTML, display
from spacy import displacy

In [378]:
def html_visiualizer(doc):
    html = displacy.render(doc, style="ent", jupyter=False)
    display(HTML(html))

### NER

In [379]:
# extract the tokens and entity tags into a dataframe
ner_df = pd.DataFrame(columns=["tokens", "labels"])
for token in spacy_doc.ents:
    ner_df = pd.concat([ner_df, pd.DataFrame.from_records([{"tokens": token.text, "labels": token.label_}])], ignore_index=True)

In [380]:
ner_df.head(10)

,tokens,labels
0,Liz Truss,PERSON
1,UK,GPE
2,Rationing,PRODUCT
3,superyachts,CARDINAL
4,Russian,NORP
5,70 years,DATE
6,70 seconds,TIME
7,Red Bull,ORG
8,Formula 1 's,PRODUCT
9,World Triathlon Championship Series,EVENT


In [381]:
# token frequency count
ner_df_counts = ner_df.groupby(["tokens", "labels"]).size().reset_index(name="counts").sort_values(by="counts", ascending=False)
ner_df_counts.head(5)

,tokens,labels,counts
965,Ukraine,GPE,47
955,UK,GPE,36
329,England,GPE,32
819,Russian,NORP,20
957,US,GPE,19


In [382]:
labels_frequency = ner_df_counts.groupby(["labels"])["tokens"].count().sort_values(ascending=False)
labels_frequency.head(15)

labels
PERSON         376
ORG            276
GPE            134
DATE            74
CARDINAL        62
NORP            49
EVENT           25
WORK_OF_ART     23
FAC             22
LOC             20
PRODUCT         16
MONEY           12
TIME             9
ORDINAL          6
PERCENT          4
Name: tokens, dtype: int64

In [383]:
# most common people
people = ner_df_counts[ner_df_counts.labels == "PERSON"]
people.head(15)

,tokens,labels,counts
257,Covid,PERSON,9
757,Putin,PERSON,8
760,Queen,PERSON,8
563,Liz Truss,PERSON,6
169,Boris Johnson,PERSON,6
788,Rishi Sunak,PERSON,5
515,Jurgen Klopp,PERSON,4
762,Quiz,PERSON,4
581,Macron,PERSON,4
325,Emma Raducanu,PERSON,4


In [384]:
# most common places
places = ner_df_counts[ner_df_counts.labels == "LOC"]
places.head(15)


,tokens,labels,counts
1005,West,LOC,2
343,Europe,LOC,2
204,Canadian Open,LOC,1
303,Earth,LOC,1
333,England v South Africa,LOC,1
341,Europa,LOC,1
427,Gulf,LOC,1
448,Hyde Park Roe,LOC,1
142,Bayern Munich,LOC,1
571,Ludwell Valley Park Tory,LOC,1


In [385]:
html_visiualizer(spacy_doc)